In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from finta import TA
from scipy.signal import find_peaks

#### Cleanse Equity Data

In [ ]:
df = pd.read_parquet("../data/raw/equities/WSM.parquet")
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")
df.reset_index(names="date", inplace=True)

In [ ]:
df.plot(
    x="date",
    y="close",
    figsize=(10, 5),
    title="Price over time",
    xlabel="date",
    ylabel="price",
    rot=0,
)
plt.show()

#### Define Reward

In [ ]:
df["fluctuation"] = 0.
peaks, _ = find_peaks(df["close"])
df.loc[peaks, "fluctuation"] = 1
bottoms, _ = find_peaks(-df["close"])
df.loc[bottoms, "fluctuation"] = -1

transition = df.query("fluctuation != 0")["fluctuation"]
for i in range(len(transition)-1):
    cur, nxt = transition.index[i:i+2]
    for j in range(cur+1, nxt):
        # step rewards
        # df.at[j, "fluctuation"] = transition.at[nxt] * (j-cur) / (nxt-cur)
        # exponential rewards
        df.at[j, "fluctuation"] = transition.at[nxt] * np.exp(-.7*(nxt-j))

#### Calculate Basic Statistics

In [ ]:
df["price"] = np.random.uniform(df["low"], df["high"]).round(2)

df["return"] = np.log1p(df["close"].pct_change())
df["return_ema20"] = df["return"].ewm(span=20).mean()
df["return_ema10"] = df["return"].ewm(span=10).mean()
df["return_ema05"] = df["return"].ewm(span=5).mean()

df["volatility_ema20"] = df["return"].ewm(span=20).std()
df["volatility_ema10"] = df["return"].ewm(span=10).std()
df["volatility_ema05"] = df["return"].ewm(span=5).std()

#### Calculate Technical Indicators

In [ ]:
df["ssma_12"] = TA.SSMA(df, 12)
df["ssma_09"] = TA.SSMA(df)
df["ssma_03"] = TA.SSMA(df, 3)

df[["macd_12_26", "signal_12_26"]] = TA.MACD(df)
df["crossover_12_26"] = df["macd_12_26"] - df["signal_12_26"]
df[["macd_06_12", "signal_06_12"]] = TA.MACD(df, 6, 12)
df["crossover_06_12"] = df["macd_06_12"] - df["signal_06_12"]

df["adx_14"] = TA.ADX(df)
df["adx_07"] = TA.ADX(df, 7)

df["rsi_14"] = TA.RSI(df)
df["rsi_07"] = TA.RSI(df, 7)

df["cci_20"] = TA.CCI(df)
df["cci_10"] = TA.CCI(df, 10)

df["bbands_20_width"] = TA.BBANDS(df).eval("BB_UPPER-BB_LOWER")
df["bbands_10_width"] = TA.BBANDS(df, 10).eval("BB_UPPER-BB_LOWER")

df["obv"] = TA.OBV(df)

df[["pivot_r1", "pivot_s1"]] = TA.PIVOT(df)[["r1", "s1"]]
df["pivot_trend_ema20"] = (df["close"]-df["pivot_r1"]).ewm(span=20).mean()
df["pivot_trend_ema10"] = (df["close"]-df["pivot_r1"]).ewm(span=10).mean()

df["atr"] = TA.ATR(df)

In [ ]:
df.dropna(inplace=True)
df.index = df["date"]
df = df[df.index.year.isin(range(2010, 2020+1))]
df = df[[
    "adx_07", "adx_14", "atr",
    "bbands_10_width", "bbands_20_width", "cci_10",
    "cci_20", "close", "crossover_06_12",
    "crossover_12_26", "fluctuation", "macd_06_12",
    "macd_12_26", "obv", "pivot_r1",
    "pivot_s1", "pivot_trend_ema10", "pivot_trend_ema20",
    "price", "return_ema05", "return_ema10",
    "return_ema20", "return", "rsi_07",
    "rsi_14", "signal_06_12", "signal_12_26",
    "ssma_03", "ssma_09", "ssma_12",
    "volatility_ema05", "volatility_ema10", "volatility_ema20",
]]